# Step 2: Data Cleaning & Preprocessing

Questo notebook pulisce e prepara i dati per l'analisi.

Operazioni:
- Gestione dei valori mancanti (NaN) tramite interpolazione
- Analisi di qualità dei dati
- Rilevamento di outlier

In [1]:
# SETUP: Configure Python path to find src/ module
# This is necessary because the notebook is in notebooks/ but src/ is at project root
import sys
from pathlib import Path

# Get the project root (parent of the notebooks directory)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # Go up one level from notebooks/

# Add project root to Python path so we can import src/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✓ Added to sys.path: {project_root}")

# Verify the path is configured
print(f"✓ Current working directory: {notebook_dir}")
print(f"✓ Project root: {project_root}")
print(f"✓ src/ module should be at: {project_root}/src")

✓ Added to sys.path: /Users/valerioquaranta/Documents/Data Science/Development/Data Visualization/project_data_manipulation
✓ Current working directory: /Users/valerioquaranta/Documents/Data Science/Development/Data Visualization/project_data_manipulation/notebooks
✓ Project root: /Users/valerioquaranta/Documents/Data Science/Development/Data Visualization/project_data_manipulation
✓ src/ module should be at: /Users/valerioquaranta/Documents/Data Science/Development/Data Visualization/project_data_manipulation/src


In [2]:
# Importa moduli
import pandas as pd
from src.data import clean_and_interpolate, get_missing_data_summary, detect_outliers_iqr
from src.config import DATA_PROCESSED_DIR

print("Modules loaded successfully!")

✓ Loaded environment variables from .env file
Modules loaded successfully!


In [3]:
# Carica il dataset grezzo dal notebook precedente
df_raw = pd.read_csv(DATA_PROCESSED_DIR / "df_raw_merged.csv")

print(f"Dataset loaded: {df_raw.shape}")
df_raw.head()

Dataset loaded: (235, 9)


,Country Name,Country Code,Year,Secondary Completion Rate (%),Education Expenditure (% of GDP),Primary Enrolment Rate(%),GDP per Capita,Life Expectancy (Years),Infant Mortality Rate (per 1000 live births)
0,Australia,AUS,2000,80.959999,4.88408,95.293571,21870.42,79.23,6.2
1,Australia,AUS,2001,81.489998,NaN,99.524277,19695.73,79.63,6.1
2,Australia,AUS,2002,82.019997,NaN,99.668716,20301.84,79.94,6.0
3,Australia,AUS,2003,82.599998,NaN,98.615700,23718.13,80.24,5.9
4,Australia,AUS,2004,83.099998,NaN,98.880447,30836.73,80.49,5.8


In [4]:
# Analisi dettagliata dei dati mancanti PRIMA della pulizia
missing_by_col, pct_missing = get_missing_data_summary(df_raw)


Missing Data Summary

Missing values by column:
Primary Enrolment Rate(%)                       94
Secondary Completion Rate (%)                   60
Education Expenditure (% of GDP)                42
GDP per Capita                                   0
Life Expectancy (Years)                          0
Infant Mortality Rate (per 1000 live births)     0
dtype: int64

Percentage of missing values by column:
Primary Enrolment Rate(%)                       40.00
Secondary Completion Rate (%)                   25.53
Education Expenditure (% of GDP)                17.87
GDP per Capita                                   0.00
Life Expectancy (Years)                          0.00
Infant Mortality Rate (per 1000 live births)     0.00
dtype: float64


Missing values by country and year:
  Australia: Years [2001, 2002, 2003, 2004, 2023]
  Brazil: Years [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2021, 2022]
  Canada: Years [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008

In [5]:
# Pulisci e interpola i dati
df_clean = clean_and_interpolate(df_raw)


Cleaning and interpolating data...

Missing values BEFORE cleaning:
Primary Enrolment Rate(%)                       94
Secondary Completion Rate (%)                   60
Education Expenditure (% of GDP)                42
GDP per Capita                                   0
Life Expectancy (Years)                          0
Infant Mortality Rate (per 1000 live births)     0
dtype: int64

  - Applying linear interpolation within each country...
  - Applying forward fill and backward fill...

Missing values AFTER cleaning:
Primary Enrolment Rate(%)                       0
Secondary Completion Rate (%)                   0
Education Expenditure (% of GDP)                0
GDP per Capita                                  0
Life Expectancy (Years)                         0
Infant Mortality Rate (per 1000 live births)    0
dtype: int64

✓ All missing values successfully handled!


In [6]:
# Verifica che non ci siano più NaN
print("\nVerifying cleaned dataset...")
print(f"\nMissing values per column:")
remaining_na = df_clean.isna().sum()
print(remaining_na[remaining_na > 0] if (remaining_na > 0).any() else "✓ No missing values!")


Verifying cleaned dataset...

Missing values per column:
✓ No missing values!


In [7]:
# Statistiche descrittive del dataset pulito
print("\nDescriptive Statistics (Cleaned Dataset):")
df_clean.describe().round(2)


Descriptive Statistics (Cleaned Dataset):


,Year,Secondary Completion Rate (%),Education Expenditure (% of GDP),Primary Enrolment Rate(%),GDP per Capita,Life Expectancy (Years),Infant Mortality Rate (per 1000 live births)
count,235.00,235.00,235.00,235.00,235.00,235.00,235.00
mean,2011.26,65.62,4.76,95.91,24065.73,74.05,22.96
std,6.81,21.93,0.93,5.74,21504.60,8.28,24.65
min,2000.00,22.43,2.70,64.96,402.94,53.91,2.80
25%,2005.00,48.79,4.11,94.18,3197.07,67.85,5.10
50%,2011.00,73.58,4.77,98.50,19695.73,77.43,8.00
75%,2017.00,85.02,5.28,99.55,43249.44,80.78,36.10
max,2023.00,95.04,7.34,99.99,77860.91,83.70,96.30


In [8]:
# Rileva outlier per ogni indicatore
from src.config import NUMERIC_COLUMNS

print("\nOutlier Detection (IQR Method)\n" + "="*60)

for col in NUMERIC_COLUMNS:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    
    if len(outliers) > 0:
        print(f"\n{col}:")
        print(f"  Bounds: [{lower:.2f}, {upper:.2f}]")
        print(f"  Outliers found: {len(outliers)}")
        for _, row in outliers.iterrows():
            print(f"    - {row['Country Name']} ({int(row['Year'])}): {row[col]:.2f}")
    else:
        print(f"\n{col}: ✓ No outliers detected")


Outlier Detection (IQR Method)

Primary Enrolment Rate(%):
  Bounds: [86.13, 107.60]
  Outliers found: 23
    - India (2000): 83.01
    - Kenya (2000): 64.96
    - Kenya (2001): 72.81
    - Kenya (2002): 70.67
    - Kenya (2003): 85.14
    - Kenya (2004): 84.26
    - Kenya (2005): 85.53
    - Kenya (2006): 85.11
    - Kenya (2012): 83.38
    - Kenya (2013): 83.38
    - Kenya (2014): 83.38
    - Kenya (2015): 83.38
    - Kenya (2016): 83.38
    - Kenya (2017): 83.38
    - Kenya (2018): 83.38
    - Kenya (2019): 83.38
    - Kenya (2020): 83.38
    - Kenya (2022): 83.38
    - South Africa (2000): 79.78
    - South Africa (2001): 81.09
    - South Africa (2002): 82.73
    - South Africa (2003): 84.38
    - South Africa (2004): 85.84

Secondary Completion Rate (%): ✓ No outliers detected

Education Expenditure (% of GDP):
  Bounds: [2.35, 7.03]
  Outliers found: 2
    - Kenya (2005): 7.34
    - Kenya (2006): 7.05

GDP per Capita: ✓ No outliers detected

Life Expectancy (Years): ✓ No outlie

In [9]:
# Salva il dataset pulito
output_file = DATA_PROCESSED_DIR / "df_clean.csv"
df_clean.to_csv(output_file, index=False)

print(f"✓ Clean dataset saved to: {output_file}")
print(f"  Shape: {df_clean.shape}")
print(f"  Columns: {df_clean.columns.tolist()}")

✓ Clean dataset saved to: /Users/valerioquaranta/Documents/Data Science/Development/Data Visualization/project_data_manipulation/data/processed/df_clean.csv
  Shape: (235, 9)
  Columns: ['Country Name', 'Country Code', 'Year', 'Secondary Completion Rate (%)', 'Education Expenditure (% of GDP)', 'Primary Enrolment Rate(%)', 'GDP per Capita', 'Life Expectancy (Years)', 'Infant Mortality Rate (per 1000 live births)']
